In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [22]:
portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')

merged = pd.read_csv('../data/all_merged.csv')
promotion = pd.read_csv('../data/promotion_df.csv')
normal = pd.read_csv('../data/normal_flow_df.csv')

---

In [23]:
display(promotion.head())
promotion.shape

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,transcript_reward,offer_type,difficulty,reward,...,gender,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,5.0,bogo,5,5,...,M,33,2017-04-21,72000.0,8.57,1,1,1,0,0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,2.0,discount,10,2,...,M,33,2017-04-21,72000.0,14.11,1,1,1,0,0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,2.0,discount,10,2,...,M,33,2017-04-21,72000.0,10.27,1,0,1,0,0
3,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,3.0,discount,7,3,...,O,40,2018-01-09,57000.0,11.93,1,1,1,1,1
4,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,5.0,bogo,5,5,...,O,40,2018-01-09,57000.0,22.05,1,1,1,1,1


(76755, 25)

In [24]:
# def get_flow_type(row):
#     viewed = row['offer viewed']
#     completed = row['offer completed']
#     is_viewed = row['is_viewed']
#     is_completed = row['is_completed']

#     if is_viewed == 1 and is_completed == 1:
#         if pd.isna(viewed) or pd.isna(completed):
#             return 0  # 타임스탬프 없으면 비정상
#         elif viewed <= completed:
#             return 1  # 받기 -> 보기 -> 완료 (정상)
#         else:
#             return 0  # 받기 -> 완료 -> 보기 (비정상)
#     elif is_viewed == 0 and is_completed == 1:
#         return 2      # 받기 -> 완료
#     elif is_viewed == 1 and is_completed == 0:
#         return 3      # 받기 -> 보기
#     else:
#         return 4      # 받기만

# promotion['flow_type'] = promotion.apply(get_flow_type, axis=1)

# # 분포 확인
# print(promotion['flow_type'].value_counts().sort_index())
def get_flow_type(row):
    viewed = row['offer viewed']
    completed = row['offer completed']
    is_viewed = row['is_viewed']
    is_completed = row['is_completed']

    if is_viewed == 1 and is_completed == 1:
        if viewed <= completed:
            return 1  # 받기 -> 보기 -> 완료 (정상)
        else:
            return 0  # 받기 -> 완료 -> 보기 (비정상)
    elif is_viewed == 0 and is_completed == 1:
        return 2      # 받기 -> 완료
    elif is_viewed == 1 and is_completed == 0:
        return 3      # 받기 -> 보기
    else:
        return 4      # 받기만

promotion['flow_type'] = promotion.apply(get_flow_type, axis=1)
counts = promotion['flow_type'].value_counts().sort_index()

print(counts)
print(f'\n총 합계: {counts.sum()}')

flow_type
0     4318
1    23519
2     5742
3    30253
4    12923
Name: count, dtype: int64

총 합계: 76755


| flow_type | 흐름 설명                     | 비고            |
|-----------|------------------------------|-----------------|
| 0         | 받기 → 완료 → 보기            | 비정상 흐름      |
| 1         | 받기 → 보기 → 완료            | 정상 흐름        |
| 2         | 받기 → 완료                   | 보기 없이 완료   |
| 3         | 받기 → 보기                   | 완료 없이 보기   |
| 4         | 받기만                        | 반응 없음        |

In [25]:
promotion.head(10)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,transcript_reward,offer_type,difficulty,reward,...,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated,flow_type
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,5.0,bogo,5,5,...,33,2017-04-21,72000.0,8.57,1,1,1,0,0,0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,2.0,discount,10,2,...,33,2017-04-21,72000.0,14.11,1,1,1,0,0,0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,2.0,discount,10,2,...,33,2017-04-21,72000.0,10.27,1,0,1,0,0,2
3,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,3.0,discount,7,3,...,40,2018-01-09,57000.0,11.93,1,1,1,1,1,1
4,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,5.0,bogo,5,5,...,40,2018-01-09,57000.0,22.05,1,1,1,1,1,1
5,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,5.0,discount,20,5,...,40,2018-01-09,57000.0,22.05,1,1,1,1,0,1
6,0020c2b971eb4e9188eac86d93036a77,bogo_10_10_5,Bogo_1,408.0,426.0,510.0,10.0,bogo,10,10,...,59,2016-03-04,90000.0,17.24,1,1,1,1,1,1
7,0020c2b971eb4e9188eac86d93036a77,discount_10_2_10,Discount_2,336.0,NaN,510.0,2.0,discount,10,2,...,59,2016-03-04,90000.0,17.24,1,0,1,0,0,2
8,0020c2b971eb4e9188eac86d93036a77,discount_10_2_10,Discount_1,0.0,12.0,54.0,2.0,discount,10,2,...,59,2016-03-04,90000.0,17.63,1,1,1,1,1,1
9,0020ccbbb6d84e358d3414a3ff76cffd,discount_7_3_7,Discount_1,168.0,168.0,222.0,3.0,discount,7,3,...,24,2016-11-11,60000.0,11.65,1,1,1,1,1,1


---

위의 표대로 계산하면 열람율 -> 전환율 -> 정상흐름비중 순으로 깔때기 모양의 퍼널이 나오게 되는 것 같다만 확실하진 않음<br>
생각해보니까 전환율과 정상흐름비중의 관계는 개념적으로 연결된 지표는 아님 -> 두 경우의 분모가 다르기 때문에

---

# 전체

In [26]:
# bogo, discount만 필터링
bd_df = promotion[promotion['offer_type'].isin(['bogo', 'discount'])].copy()

In [27]:
normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()
viewed_bd_df = bd_df[bd_df['flow_type'].isin([1, 3])].copy()
invalid_bd_df = bd_df[bd_df['flow_type'] == 0].copy()
completed_only_bd_df = bd_df[bd_df['flow_type'] == 2].copy()

def get_funnel_dict(total, viewed, normal, invalid, completed_only):
    return {
        'total_received': len(total),
        'total_viewed': total['is_viewed'].sum(),
        'total_completed': total['is_completed'].sum(),
        'normal_flow': len(normal),
        '열람율(%)': round(viewed['is_viewed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
        '단계별전환율(%)': round(len(normal) / len(viewed) * 100, 2) if len(viewed) > 0 else 0,
        '정상흐름비중(%)': round(len(normal) / len(total) * 100, 2) if len(total) > 0 else 0,
        '비정상흐름비중(%)': round(len(invalid) / len(total) * 100, 2) if len(total) > 0 else 0,
        '바로완료비중(%)': round(len(completed_only) / len(total) * 100, 2) if len(total) > 0 else 0,
        '완료율(%)': round(total['is_completed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
    }

# 전체
overall_funnel = pd.DataFrame([{
    'group': 'overall',
    **get_funnel_dict(bd_df, viewed_bd_df, normal_bd_df, invalid_bd_df, completed_only_bd_df)
}])
display(overall_funnel)

# offer_type
type_funnel_list = []
for offer_type, group in bd_df.groupby('offer_type'):
    viewed = viewed_bd_df[viewed_bd_df['offer_type'] == offer_type]
    normal = normal_bd_df[normal_bd_df['offer_type'] == offer_type]
    invalid = invalid_bd_df[invalid_bd_df['offer_type'] == offer_type]
    completed_only = completed_only_bd_df[completed_only_bd_df['offer_type'] == offer_type]
    type_funnel_list.append({
        'offer_type': offer_type,
        **get_funnel_dict(group, viewed, normal, invalid, completed_only)
    })

type_funnel = pd.DataFrame(type_funnel_list)
display(type_funnel)

# offer_id
id_funnel_list = []
for offer_id, group in bd_df.groupby('offer_id'):
    viewed = viewed_bd_df[viewed_bd_df['offer_id'] == offer_id]
    normal = normal_bd_df[normal_bd_df['offer_id'] == offer_id]
    invalid = invalid_bd_df[invalid_bd_df['offer_id'] == offer_id]
    completed_only = completed_only_bd_df[completed_only_bd_df['offer_id'] == offer_id]
    id_funnel_list.append({
        'offer_id': offer_id,
        **get_funnel_dict(group, viewed, normal, invalid, completed_only)
    })

id_funnel = pd.DataFrame(id_funnel_list)
display(id_funnel)

,group,total_received,total_viewed,total_completed,normal_flow,열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,overall,61520,47259,33579,23519,69.8,54.77,38.23,7.02,9.33,54.58


,offer_type,total_received,total_viewed,total_completed,normal_flow,열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,bogo,30667,25588,15669,11018,75.69,47.47,35.93,7.75,7.42,51.09
1,discount,30853,21671,17910,12501,63.95,63.36,40.52,6.29,11.24,58.05


,offer_id,total_received,total_viewed,total_completed,normal_flow,열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,bogo_10_10_5,7623,7326,3331,2759,89.81,40.30,36.19,6.30,1.21,43.70
1,bogo_10_10_7,7711,6763,3688,2606,79.29,42.62,33.80,8.42,5.62,47.83
2,bogo_5_5_5,7605,7298,4296,3529,87.59,52.98,46.40,8.38,1.71,56.49
3,bogo_5_5_7,7728,4201,4354,2124,46.45,59.16,27.48,7.91,20.95,56.34
4,discount_10_2_10,7684,7412,5317,4643,89.55,67.48,60.42,6.91,1.86,69.20
5,discount_10_2_7,7694,4160,4017,2134,46.98,59.03,27.74,7.08,17.39,52.21
6,discount_20_5_10,7775,2711,3420,1336,31.58,54.42,17.18,3.29,23.51,43.99
7,discount_7_3_7,7700,7388,5156,4388,88.04,64.73,56.99,7.91,2.06,66.96


---

# 채널별

In [28]:
channels = ['web', 'email', 'mobile', 'social']

normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()
viewed_bd_df = bd_df[bd_df['flow_type'].isin([1, 3])].copy()
invalid_bd_df = bd_df[bd_df['flow_type'] == 0].copy()
completed_only_bd_df = bd_df[bd_df['flow_type'] == 2].copy()

# 채널 전체
channel_funnel_list = []
for channel in channels:
    total = bd_df[bd_df[channel] == 1]
    viewed = viewed_bd_df[viewed_bd_df[channel] == 1]
    normal = normal_bd_df[normal_bd_df[channel] == 1]
    invalid = invalid_bd_df[invalid_bd_df[channel] == 1]
    completed_only = completed_only_bd_df[completed_only_bd_df[channel] == 1]
    channel_funnel_list.append({
        'channel': channel,
        'total_received': len(total),
        'total_viewed': total['is_viewed'].sum(),
        'total_completed': total['is_completed'].sum(),
        'normal_flow': len(normal),
        '열람율(%)': round(viewed['is_viewed'].sum() / len(total) * 100, 2),
        # '전환율(%)': round(viewed['is_completed'].sum() / len(total) * 100, 2),
        '단계별전환율(%)': round(len(normal) / len(viewed) * 100, 2) if len(viewed) > 0 else 0,
        '정상흐름비중(%)': round(len(normal) / len(total) * 100, 2),
        '비정상흐름비중(%)': round(len(invalid) / len(total) * 100, 2),
        '바로완료비중(%)': round(len(completed_only) / len(total) * 100, 2),
        '완료율(%)': round(total['is_completed'].sum() / len(total) * 100, 2),
    })

channel_funnel = pd.DataFrame(channel_funnel_list)
display(channel_funnel)

# 채널 x offer_type
channel_type_list = []
for channel in channels:
    total_temp = bd_df[bd_df[channel] == 1]
    viewed_temp = viewed_bd_df[viewed_bd_df[channel] == 1]
    normal_temp = normal_bd_df[normal_bd_df[channel] == 1]
    invalid_temp = invalid_bd_df[invalid_bd_df[channel] == 1]
    completed_only_temp = completed_only_bd_df[completed_only_bd_df[channel] == 1]
    for offer_type in total_temp['offer_type'].unique():
        total_group = total_temp[total_temp['offer_type'] == offer_type]
        viewed_group = viewed_temp[viewed_temp['offer_type'] == offer_type]
        normal_group = normal_temp[normal_temp['offer_type'] == offer_type]
        invalid_group = invalid_temp[invalid_temp['offer_type'] == offer_type]
        completed_only_group = completed_only_temp[completed_only_temp['offer_type'] == offer_type]
        channel_type_list.append({
            'channel': channel,
            'offer_type': offer_type,
            'total_received': len(total_group),
            'total_viewed': total_group['is_viewed'].sum(),
            'total_completed': total_group['is_completed'].sum(),
            'normal_flow': len(normal_group),
            '열람율(%)': round(viewed_group['is_viewed'].sum() / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            # '전환율(%)': round(viewed_group['is_completed'].sum() / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            '단계별전환율(%)': round(len(normal_group) / len(viewed_group) * 100, 2) if len(viewed_group) > 0 else 0,
            '정상흐름비중(%)': round(len(normal_group) / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            '비정상흐름비중(%)': round(len(invalid_group) / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            '바로완료비중(%)': round(len(completed_only_group) / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            '완료율(%)': round(total_group['is_completed'].sum() / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
        })

channel_type_funnel = pd.DataFrame(channel_type_list)
display(channel_type_funnel)

# 채널 x offer_id
channel_id_list = []
for channel in channels:
    total_temp = bd_df[bd_df[channel] == 1]
    viewed_temp = viewed_bd_df[viewed_bd_df[channel] == 1]
    normal_temp = normal_bd_df[normal_bd_df[channel] == 1]
    invalid_temp = invalid_bd_df[invalid_bd_df[channel] == 1]
    completed_only_temp = completed_only_bd_df[completed_only_bd_df[channel] == 1]
    for offer_id in total_temp['offer_id'].unique():
        total_group = total_temp[total_temp['offer_id'] == offer_id]
        viewed_group = viewed_temp[viewed_temp['offer_id'] == offer_id]
        normal_group = normal_temp[normal_temp['offer_id'] == offer_id]
        invalid_group = invalid_temp[invalid_temp['offer_id'] == offer_id]
        completed_only_group = completed_only_temp[completed_only_temp['offer_id'] == offer_id]
        channel_id_list.append({
            'channel': channel,
            'offer_id': offer_id,
            'total_received': len(total_group),
            'total_viewed': total_group['is_viewed'].sum(),
            'total_completed': total_group['is_completed'].sum(),
            'normal_flow': len(normal_group),
            '열람율(%)': round(viewed_group['is_viewed'].sum() / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            # '전환율(%)': round(viewed_group['is_completed'].sum() / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            '단계별전환율(%)': round(len(normal_group) / len(viewed_group) * 100, 2) if len(viewed_group) > 0 else 0,
            '정상흐름비중(%)': round(len(normal_group) / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            '비정상흐름비중(%)': round(len(invalid_group) / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            '바로완료비중(%)': round(len(completed_only_group) / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
            '완료율(%)': round(total_group['is_completed'].sum() / len(total_group) * 100, 2) if len(total_group) > 0 else 0,
        })

channel_id_funnel = pd.DataFrame(channel_id_list)
display(channel_id_funnel)

,channel,total_received,total_viewed,total_completed,normal_flow,열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,web,53809,40496,29891,20913,68.44,56.79,38.87,6.82,9.87,55.55
1,email,61520,47259,33579,23519,69.80,54.77,38.23,7.02,9.33,54.58
2,mobile,53745,44548,30159,22183,75.33,54.79,41.27,7.56,7.28,56.11
3,social,38323,36187,21788,17925,86.84,53.86,46.77,7.58,2.50,56.85


,channel,offer_type,total_received,total_viewed,total_completed,normal_flow,열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,web,bogo,22956,18825,11981,8412,74.48,49.20,36.64,7.53,8.02,52.19
1,web,discount,30853,21671,17910,12501,63.95,63.36,40.52,6.29,11.24,58.05
2,email,bogo,30667,25588,15669,11018,75.69,47.47,35.93,7.75,7.42,51.09
3,email,discount,30853,21671,17910,12501,63.95,63.36,40.52,6.29,11.24,58.05
4,mobile,bogo,30667,25588,15669,11018,75.69,47.47,35.93,7.75,7.42,51.09
5,mobile,discount,23078,18960,14490,11165,74.85,64.63,48.38,7.30,7.11,62.79
6,social,bogo,22939,21387,11315,8894,85.54,45.33,38.77,7.70,2.86,49.33
7,social,discount,15384,14800,10473,9031,88.79,66.11,58.70,7.41,1.96,68.08


,channel,offer_id,total_received,total_viewed,total_completed,normal_flow,열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,web,bogo_5_5_5,7605,7298,4296,3529,87.59,52.98,46.40,8.38,1.71,56.49
1,web,discount_10_2_10,7684,7412,5317,4643,89.55,67.48,60.42,6.91,1.86,69.20
2,web,discount_10_2_7,7694,4160,4017,2134,46.98,59.03,27.74,7.08,17.39,52.21
3,web,discount_7_3_7,7700,7388,5156,4388,88.04,64.73,56.99,7.91,2.06,66.96
4,web,bogo_5_5_7,7728,4201,4354,2124,46.45,59.16,27.48,7.91,20.95,56.34
5,web,discount_20_5_10,7775,2711,3420,1336,31.58,54.42,17.18,3.29,23.51,43.99
6,web,bogo_10_10_5,7623,7326,3331,2759,89.81,40.30,36.19,6.30,1.21,43.70
7,email,bogo_5_5_5,7605,7298,4296,3529,87.59,52.98,46.40,8.38,1.71,56.49
8,email,discount_10_2_10,7684,7412,5317,4643,89.55,67.48,60.42,6.91,1.86,69.20
9,email,discount_10_2_7,7694,4160,4017,2134,46.98,59.03,27.74,7.08,17.39,52.21


---

# 채널 조합별

In [29]:
# 채널 조합 컬럼 생성
def get_channel_combo(row):
    channels = []
    for ch in ['web', 'email', 'mobile', 'social']:
        if row[ch] == 1:
            channels.append(ch)
    return '+'.join(channels) if channels else 'none'

bd_df['channel_combo'] = bd_df.apply(get_channel_combo, axis=1)

normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()
viewed_bd_df = bd_df[bd_df['flow_type'].isin([1, 3])].copy()
invalid_bd_df = bd_df[bd_df['flow_type'] == 0].copy()
completed_only_bd_df = bd_df[bd_df['flow_type'] == 2].copy()

# 채널 조합 전체
combo_funnel_list = []
for combo, group in bd_df.groupby('channel_combo'):
    viewed_group = viewed_bd_df[viewed_bd_df['channel_combo'] == combo]
    normal_group = normal_bd_df[normal_bd_df['channel_combo'] == combo]
    invalid_group = invalid_bd_df[invalid_bd_df['channel_combo'] == combo]
    completed_only_group = completed_only_bd_df[completed_only_bd_df['channel_combo'] == combo]
    combo_funnel_list.append({
        'channel_combo': combo,
        'total_received': len(group),
        'total_viewed': group['is_viewed'].sum(),
        'total_completed': group['is_completed'].sum(),
        'normal_flow': len(normal_group),
        '열람율(%)': round(viewed_group['is_viewed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '전환율(%)': round(viewed_group['is_completed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '단계별전환율(%)': round(len(normal_group) / len(viewed_group) * 100, 2) if len(viewed_group) > 0 else 0,
        '완료율(%)': round(group['is_completed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '정상흐름비중(%)': round(len(normal_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '비정상흐름비중(%)': round(len(invalid_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '바로완료비중(%)': round(len(completed_only_group) / len(group) * 100, 2) if len(group) > 0 else 0,
    })

combo_funnel = pd.DataFrame(combo_funnel_list).sort_values('total_received', ascending=False)
display(combo_funnel)

# 채널 조합 x offer_type
combo_type_list = []
for (combo, offer_type), group in bd_df.groupby(['channel_combo', 'offer_type']):
    viewed_group = viewed_bd_df[(viewed_bd_df['channel_combo'] == combo) & (viewed_bd_df['offer_type'] == offer_type)]
    normal_group = normal_bd_df[(normal_bd_df['channel_combo'] == combo) & (normal_bd_df['offer_type'] == offer_type)]
    invalid_group = invalid_bd_df[(invalid_bd_df['channel_combo'] == combo) & (invalid_bd_df['offer_type'] == offer_type)]
    completed_only_group = completed_only_bd_df[(completed_only_bd_df['channel_combo'] == combo) & (completed_only_bd_df['offer_type'] == offer_type)]
    combo_type_list.append({
        'channel_combo': combo,
        'offer_type': offer_type,
        'total_received': len(group),
        'total_viewed': group['is_viewed'].sum(),
        'total_completed': group['is_completed'].sum(),
        'normal_flow': len(normal_group),
        '열람율(%)': round(viewed_group['is_viewed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '전환율(%)': round(viewed_group['is_completed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '단계별전환율(%)': round(len(normal_group) / len(viewed_group) * 100, 2) if len(viewed_group) > 0 else 0,
        '완료율(%)': round(group['is_completed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '정상흐름비중(%)': round(len(normal_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '비정상흐름비중(%)': round(len(invalid_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '바로완료비중(%)': round(len(completed_only_group) / len(group) * 100, 2) if len(group) > 0 else 0,
    })

combo_type_funnel = pd.DataFrame(combo_type_list).sort_values('total_received', ascending=False)
display(combo_type_funnel)

# 채널 조합 x offer_id
combo_id_list = []
for (combo, offer_id), group in bd_df.groupby(['channel_combo', 'offer_id']):
    viewed_group = viewed_bd_df[(viewed_bd_df['channel_combo'] == combo) & (viewed_bd_df['offer_id'] == offer_id)]
    normal_group = normal_bd_df[(normal_bd_df['channel_combo'] == combo) & (normal_bd_df['offer_id'] == offer_id)]
    invalid_group = invalid_bd_df[(invalid_bd_df['channel_combo'] == combo) & (invalid_bd_df['offer_id'] == offer_id)]
    completed_only_group = completed_only_bd_df[(completed_only_bd_df['channel_combo'] == combo) & (completed_only_bd_df['offer_id'] == offer_id)]
    combo_id_list.append({
        'channel_combo': combo,
        'offer_id': offer_id,
        'total_received': len(group),
        'total_viewed': group['is_viewed'].sum(),
        'total_completed': group['is_completed'].sum(),
        'normal_flow': len(normal_group),
        '열람율(%)': round(viewed_group['is_viewed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '전환율(%)': round(viewed_group['is_completed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '단계별전환율(%)': round(len(normal_group) / len(viewed_group) * 100, 2) if len(viewed_group) > 0 else 0,
        '완료율(%)': round(group['is_completed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '정상흐름비중(%)': round(len(normal_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '비정상흐름비중(%)': round(len(invalid_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '바로완료비중(%)': round(len(completed_only_group) / len(group) * 100, 2) if len(group) > 0 else 0,
    })

combo_id_funnel = pd.DataFrame(combo_id_list).sort_values('total_received', ascending=False)
display(combo_id_funnel)

,channel_combo,total_received,total_viewed,total_completed,normal_flow,열람율(%),전환율(%),단계별전환율(%),완료율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%)
3,web+email+mobile+social,30612,29424,18100,15319,88.75,50.04,56.39,59.13,50.04,7.37,1.71
2,web+email+mobile,15422,8361,8371,4258,46.72,27.61,59.10,54.28,27.61,7.50,19.17
1,web+email,7775,2711,3420,1336,31.58,17.18,54.42,43.99,17.18,3.29,23.51
0,email+mobile+social,7711,6763,3688,2606,79.29,33.80,42.62,47.83,33.80,8.42,5.62


,channel_combo,offer_type,total_received,total_viewed,total_completed,normal_flow,열람율(%),전환율(%),단계별전환율(%),완료율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%)
5,web+email+mobile+social,discount,15384,14800,10473,9031,88.79,58.70,66.11,68.08,58.70,7.41,1.96
4,web+email+mobile+social,bogo,15228,14624,7627,6288,88.70,41.29,46.55,50.09,41.29,7.34,1.46
1,web+email,discount,7775,2711,3420,1336,31.58,17.18,54.42,43.99,17.18,3.29,23.51
2,web+email+mobile,bogo,7728,4201,4354,2124,46.45,27.48,59.16,56.34,27.48,7.91,20.95
0,email+mobile+social,bogo,7711,6763,3688,2606,79.29,33.80,42.62,47.83,33.80,8.42,5.62
3,web+email+mobile,discount,7694,4160,4017,2134,46.98,27.74,59.03,52.21,27.74,7.08,17.39


,channel_combo,offer_id,total_received,total_viewed,total_completed,normal_flow,열람율(%),전환율(%),단계별전환율(%),완료율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%)
1,web+email,discount_20_5_10,7775,2711,3420,1336,31.58,17.18,54.42,43.99,17.18,3.29,23.51
2,web+email+mobile,bogo_5_5_7,7728,4201,4354,2124,46.45,27.48,59.16,56.34,27.48,7.91,20.95
0,email+mobile+social,bogo_10_10_7,7711,6763,3688,2606,79.29,33.80,42.62,47.83,33.80,8.42,5.62
7,web+email+mobile+social,discount_7_3_7,7700,7388,5156,4388,88.04,56.99,64.73,66.96,56.99,7.91,2.06
3,web+email+mobile,discount_10_2_7,7694,4160,4017,2134,46.98,27.74,59.03,52.21,27.74,7.08,17.39
6,web+email+mobile+social,discount_10_2_10,7684,7412,5317,4643,89.55,60.42,67.48,69.20,60.42,6.91,1.86
4,web+email+mobile+social,bogo_10_10_5,7623,7326,3331,2759,89.81,36.19,40.30,43.70,36.19,6.30,1.21
5,web+email+mobile+social,bogo_5_5_5,7605,7298,4296,3529,87.59,46.40,52.98,56.49,46.40,8.38,1.71


| flow_type | 흐름 설명                     | 비고            |
|-----------|------------------------------|-----------------|
| 0         | 받기 → 완료 → 보기            | 비정상 흐름      |
| 1         | 받기 → 보기 → 완료            | 정상 흐름        |
| 2         | 받기 → NaN -> 완료                   | 바로 완료   |
| 3         | 받기 → 보기 -> NaN                   | 완료 없이 보기   |
| 4         | 받기 -> NaN -> NaN                        | 반응 없음        |

| 지표     | 분자                                   | 분모                                   |
|----------|----------------------------------------|----------------------------------------|
| 열람율   | flow_type.isin([1,3])의 is_viewed       | flow_type.isin([0,1,2,3,4])의 count      |
| 전환율   | flow_type.isin([1,3])의 is_completed    | flow_type.isin([0,1,2,3,4])의 count      |
| 단계별전환율 | flow_type.isin([1])의 is_completed | flow_type.isin([1,3])의 count |
| 완료율   | flow_type.isin([0,1,2,3,4])의 is_completed | flow_type.isin([0,1,2,3,4])의 count |
| 정상흐름비중 | flow_type == 1의 count | flow_type.isin([0,1,2,3,4])의 count |
| 비정상흐름비중 | flow_type == 0의 count | flow_type.isin([0,1,2,3,4])의 count |
| 바로완료비중 | flow_type == 2의 count | flow_type.isin([0,1,2,3,4])의 count |

1. 전체: recieved
2. 열람: 전체 중 `받기 -> 보기`가 있는 경우
3. 완료: 전체 중 `받기 -> 보기 -> 완료`가 있는 경우<br>
-> 완료율이 정상흐름 완료율과 같이짐
=> 퍼널 흐름?